In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from src.data import load_train
from src.config import FAMILIES, SEED
from src.evaluation import get_cv, multiclass_metrics, binary_view, evaluate_cv, save_results
from src.models import get_models

X_train, y_train, meta_train = load_train()
print(f"X_train: {X_train.shape}  | families: {dict(y_train.value_counts())}")

In [ ]:
#            0        1        2      3      4        5       6        7      8        9
y_true = ['Normal','Normal','Normal','DoS','DoS','Probe','R2L','U2R','Normal','DoS']
y_pred = ['Normal','Normal','DoS',  'DoS','Normal','Probe','Normal','U2R','Normal','DoS']

# Hand computation:
#   Actual Normal = {0,1,2,8} (4); actual attacks = {3,4,5,6,7,9} (6)
#   FPR = Normal->attack / Normal = 1/4 = 0.25   (row 2)
#   FNR = attack->Normal / attacks = 2/6 = 1/3   (rows 4,6)
#   Per-class F1: Normal=2/3, DoS=2/3, Probe=1, R2L=0, U2R=1  -> macro = 2/3
m = multiclass_metrics(y_true, y_pred, labels=FAMILIES)
b = binary_view(y_true, y_pred)

assert np.isclose(m["macro_f1"], 2/3), f"macro_f1 {m['macro_f1']}"
assert np.isclose(b["fpr"], 0.25), f"fpr {b['fpr']}"
assert np.isclose(b["fnr"], 1/3), f"fnr {b['fnr']}"

# Cross-check macro-F1 against sklearn directly
from sklearn.metrics import f1_score
sk = f1_score(y_true, y_pred, average="macro", labels=FAMILIES, zero_division=0)
assert np.isclose(m["macro_f1"], sk)
print("Test 1: hand-computed macro-F1/FPR/FNR match")

In [ ]:
y5 = ["Normal", "DoS", "Probe", "R2L", "U2R"]

# Perfect predictions
mp = multiclass_metrics(y5, y5, labels=FAMILIES)
bp = binary_view(y5, y5)
assert np.isclose(mp["macro_f1"], 1.0) and np.isclose(bp["fpr"], 0.0) and np.isclose(bp["fnr"], 0.0)

# All-Normal predictions -> FPR 0, FNR 1
bn = binary_view(y5, ["Normal"] * 5)
assert np.isclose(bn["fpr"], 0.0) and np.isclose(bn["fnr"], 1.0)

# All-attack predictions -> FPR 1
ba = binary_view(y5, ["DoS"] * 5)
assert np.isclose(ba["fpr"], 1.0)
print("Test 2: perfect / all-Normal / all-attack edge cases pass")

In [ ]:
cv1, cv2 = get_cv(), get_cv()
s1 = list(cv1.split(X_train, y_train))
s2 = list(cv2.split(X_train, y_train))

for (tr1, va1), (tr2, va2) in zip(s1, s2):
    assert np.array_equal(tr1, tr2) and np.array_equal(va1, va2), "CV not reproducible"

for tr, va in s1:
    assert len(set(tr) & set(va)) == 0, "train/val overlap within a fold"

all_val = np.concatenate([va for _, va in s1])
assert len(all_val) == len(X_train) and len(set(all_val)) == len(X_train), \
    "OOF coverage broken"
print("Test 3: get_cv reproducible; folds disjoint; every row in exactly one val fold")

In [ ]:
models = get_models()
cv = get_cv()

for name, pipe in models.items():
    fold_df, oof = evaluate_cv(pipe, X_train, y_train, cv)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {fold_df['accuracy'].mean():.4f}   (expected ~0.60)")
    print(f"Macro-F1 : {fold_df['macro_f1'].mean():.4f}   (expected ~0.15)")
    print(f"FPR      : {fold_df['fpr'].mean():.4f}   (expected 0.0)")
    print(f"FNR      : {fold_df['fnr'].mean():.4f}   (expected 1.0)")
    for fam in ["DoS", "Probe", "R2L", "U2R"]:
        print(f"  recall_{fam:<5}: {fold_df[f'recall_{fam}'].mean():.4f}   (expected 0.0)")

    path = save_results(name, fold_df)
    print(f"saved -> {path}")

In [ ]:
fold_df, oof = evaluate_cv(models["M0_dummy"], X_train, y_train, get_cv())

assert len(oof) == len(X_train), "OOF length mismatch"
assert set(oof) <= set(FAMILIES), "OOF contains unexpected labels"
assert set(oof) == {"Normal"}, "Dummy(most_frequent) should predict only Normal"

assert np.isclose(fold_df["fpr"].mean(), 0.0)
assert np.isclose(fold_df["fnr"].mean(), 1.0)
for fam in ["DoS", "Probe", "R2L", "U2R"]:
    assert np.isclose(fold_df[f"recall_{fam}"].mean(), 0.0)
print("Dummy behaves exactly as theory predicts — harness is trustworthy")

Phase 9 cells

In [ ]:
import warnings
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.base import clone


from src.config import FAMILIES, SEED
from src.evaluation import get_cv, evaluate_cv, save_results
from src.models import get_models
from src.preprocessing import build_pipeline
from sklearn.linear_model import LogisticRegression



In [ ]:
weights = compute_class_weight("balanced", classes=np.array(FAMILIES), y=y_train.values)
print("class_weight='balanced' weights  =  n / (k * n_class):")
for fam, w in zip(FAMILIES, weights):
    print(f"  {fam:6s}: weight={w:8.1f}   (train n={int((y_train == fam).sum())})")

In [ ]:
X_s, X_v, y_s, y_v = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)

pipe = clone(get_models()["M1_logreg"])
pipe.fit(X_s, y_s)

clf = pipe.named_steps["clf"]
assert set(clf.classes_) == set(FAMILIES), "classes_ missing a family"
print("classes_:", list(clf.classes_))

proba = pipe.predict_proba(X_v)
assert np.allclose(proba.sum(axis=1), 1.0), "predict_proba rows don't sum to 1"
print("predict_proba rows sum to 1 ")

macro = f1_score(y_v, pipe.predict(X_v), average="macro", labels=FAMILIES, zero_division=0)
print(f"Held-out slice macro-F1: {macro:.4f}  (Dummy is ~0.15)")
assert macro > 0.15, "LR should beat the Dummy easily"
print(" sanity check: LR learns, classes and probabilities are well-formed")


In [ ]:
pipe_noscale = build_pipeline(
    LogisticRegression(solver="lbfgs", class_weight="balanced", max_iter=100, random_state=SEED),
    scale=False, engineered=False
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    pipe_noscale.fit(X_s, y_s)
    conv = [w for w in caught if "onverg" in str(w.message).lower()]
    print(f"ConvergenceWarnings (unscaled): {len(conv)}")
    if conv:
        print("  e.g.:", str(conv[0].message)[:140])


In [ ]:
cv = get_cv()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fold_df, oof = evaluate_cv(get_models()["M1_logreg"], X_train, y_train, cv)
    conv = [w for w in caught if "onverg" in str(w.message).lower()]

print(f"ConvergenceWarnings during M1 CV: {len(conv)}")
if conv:
    print("  first:", str(conv[0].message)[:140])

print(f"\n=== M1_logreg (5-fold CV) ===")
print(f"Macro-F1 : {fold_df['macro_f1'].mean():.4f} ± {fold_df['macro_f1'].std():.4f}")
print(f"FPR      : {fold_df['fpr'].mean():.4f}   (limit 0.005)")
print(f"FNR      : {fold_df['fnr'].mean():.4f}")
for fam in FAMILIES:
    print(f"  recall_{fam:<6}: {fold_df[f'recall_{fam}'].mean():.4f}")
print(f"Avg fit_time: {fold_df['fit_time'].mean():.1f}s")

path = save_results("M1_logreg", fold_df)
print(f"saved -> {path}")

In [ ]:
feat_names = pipe.named_steps["pre"].get_feature_names_out()  # pipe is fitted (Cell 3)
coef = pipe.named_steps["clf"].coef_       # shape (n_classes, n_features)
classes = pipe.named_steps["clf"].classes_

for i, cls in enumerate(classes):
    top = np.argsort(coef[i])[-8:][::-1]   # 8 strongest positive drivers of this class
    print(f"\n=== {cls} ===")
    for j in top:
        print(f"  {coef[i, j]:+9.3f}  {feat_names[j]}")

In [ ]:
pipe_nocw = build_pipeline(
    LogisticRegression(solver="lbfgs", max_iter=3000, random_state=SEED),  # no class_weight
    scale=True, engineered=False,
)
fold_nocw, _ = evaluate_cv(pipe_nocw, X_train, y_train, get_cv())
print(f"NO class_weight: macro-F1={fold_nocw['macro_f1'].mean():.4f}  "
      f"FPR={fold_nocw['fpr'].mean():.4f}")
print(f"  recall_R2L={fold_nocw['recall_R2L'].mean():.4f}  "
      f"recall_U2R={fold_nocw['recall_U2R'].mean():.4f}")
print(f"WITH class_weight (M1): macro-F1={fold_df['macro_f1'].mean():.4f}  "
      f"FPR={fold_df['fpr'].mean():.4f}")